# Part I: Portfolio Optimization

## Problem Overview

Suppose you have some money to invest. How should you divide up your money across investments to maximize your gains? One approach is to invest all your money in the stock with the highest historical growth. But that strategy ignores risk: a stock that grows a lot might also fluctuate a lot. Maybe by investing in different stocks, some volatile and some stable, you can avoid taking on too much risk while still achieving a fairly high return.

In this homework, we provide you with a data set of historical stock prices, i.e., for various stocks, their price progressions over time. Your goal is to formulate a **mathematical program** that finds a portfolio among the stocks in the data set that maximizes total return while keeping risk below a specified threshold. You will then implement your mathematical program using `CVXPY`.

## Load Necessary Imports

In [ ]:
import sys
from pathlib import Path
from typing import Dict, List, Tuple, Union

if sys.version_info[:3] != (3, 10, 6):
    raise RuntimeError("Select the Python 3.10.6 (CS410) kernel before continuing.")

import cvxpy as cvx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

candidates = [
    Path("World-Stock-Prices-Dataset.csv"),
    Path("../World-Stock-Prices-Dataset.csv"),
    Path("brown-cs410-f26-constrained-optimization-assignment-5/World-Stock-Prices-Dataset.csv"),
]
DATA_FILE = next((path for path in candidates if path.is_file()), None)

## The Data

We have provided you with historical stock price data in `World-Stock-Prices-Dataset.csv`, which reports stock price data over time.

Before you can build an optimization model in `CVXPY`, you need to import the raw CSV into a Python data structure that is compatible with `CVXPY`, such as a dictionary mapping each stock name to its corresponding historical observations:

```python 
{
    "AAPL": [1.00, 1.01, 0.99, ..., 1.24],
    "MSFT": [1.00, 0.98, 1.02, ..., 1.18],
    ...
}
```

We normalize each stock's prices so that its initial price is $1$, which allows you to compare the growth rates of stocks with different initial prices.

For example, if a stock's normalized price starts at $1$ and ends at $1.25$, then an initial investment of $5$ would be worth $6.25$ at the end of the period, representing a 25% return.

In this homework, we define a stock's return as its historical return: $\text{final normalized price} - 1$. For a stock with observed historical prices $x_1,x_2,\ldots,x_m$, the return is $x_m - 1$.

#### Stock Choices

We choose a list of only 10 stocks to optimize over, so that portfolio optimization is fast. You should consider only these ten when developing your model, so that your code passes our provided tests. But you should also feel free to experiment with alternative stock choices once your model is fully developed.

In [ ]:
stock_names = sorted(["AAPL", "JNJ", "JPM", "KO", "MSFT", "NVDA", "PG", "SPOT", "TSLA", "V"])


Run the cell below to preview the raw CSV, pre-process the stock data, and display the normalized price histories used in this homework.

In [ ]:
# DO NOT EDIT

print("This notebook uses the following dataset:")
raw_prices = pd.read_csv(DATA_FILE)
display(raw_prices.head())


def load_data(filename: Union[str, Path]) -> pd.DataFrame:
    """Read the latest year of prices for tickers with at least 252 observations."""
    df = pd.read_csv(filename)
    numeric_columns = [
        "Open", "High", "Low", "Close", "Volume", "Dividends", "Stock Splits"
    ]
    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")
    df["Date"] = pd.to_datetime(df["Date"], utc=True)
    cutoff = df["Date"].max() - pd.DateOffset(years=1)
    df = df[df["Date"] >= cutoff]
    df = df.drop_duplicates(subset=["Date", "Ticker"], keep="first")
    df = df.groupby("Ticker").filter(lambda group: len(group) >= 252)
    return df.sort_values(["Ticker", "Date"])


def normalize_stock_data(stock_df: pd.DataFrame) -> pd.DataFrame:
    """Normalize each ticker's closing prices to its first observation."""
    df = stock_df.copy().sort_values(["Ticker", "Date"])
    first_prices = df.groupby("Ticker")["Close"].transform("first")
    df["Normalized_Close"] = df["Close"] / first_prices
    return df


def create_price_dictionary(df: pd.DataFrame) -> Dict[str, List[float]]:
    """Return normalized price histories keyed by ticker."""
    normalized = normalize_stock_data(df)
    return {
        str(ticker): group.sort_values("Date")["Normalized_Close"].tolist()
        for ticker, group in normalized.groupby("Ticker")
    }


def get_data() -> Dict[str, List[float]]:
    """Return normalized historical prices from the supplied dataset."""
    return create_price_dictionary(load_data(DATA_FILE))


data = get_data()

print("VIEW YOUR PRE-PROCESSED DATA BELOW!\nRows are the stock price time periods, and the columns are the normalized prices for each stock.\n")
print("First 5 observations for the supplied stocks:")
display(pd.DataFrame({ticker: data[ticker] for ticker in stock_names}).head())
print("Last 5 observations for the supplied stocks:")
display(pd.DataFrame({ticker: data[ticker] for ticker in stock_names}).tail())


#### Risk

In this homework, you will use **variance** to model risk. The variance of a random variable $X$ captures how spread out its values are around its expected value. It is defined as:

$$\textrm{Var}(X) =\mathbb{E} \left[ (X - \mathbb{E}(X))^2 \right]$$

In portfolio optimization, variance captures how much a stock's historical price fluctuates around its average. Using squared deviations from the expected value, rather than the deviations themselves, makes sense because positive and negative deviations would otherwise cancel each other out. Moreover, squaring deviations assigns disproportionately greater weight to large fluctuations, reflecting the intuition that substantial deviations from expected returns can have a much greater financial impact than small ones. (However, variance penalizes both unexpectedly low and unexpectedly high returns equally.)

In a portfolio with multiple stocks, the risk of the portfolio depends not just on each individual stock's variance, but also on how their returns move together. **Covariance**, which generalizes variance, captures these movements. The **covariance** of two random variables $Y$ and $Z$ is defined as follows:

$$\textrm{Cov}(Y, Z) = \mathbb{E} \left[ (Y - \mathbb{E} \left[ Y \right]) (Z - \mathbb{E} \left[ Z \right]) \right]$$

Positive covariance means $X$ and $Y$ tend to move in the same direction relative to their expected values; negative covariance means they tend to move in opposite directions. Note that $\textrm{Cov}(X, X) = \textrm{Var}(X)$.

We can collect the variances and covariances of a set of stocks into a covariance matrix. Covariance is often denoted by $\Sigma$; note, however, $\Sigma$ is a matrix, not the summation symbol! For two stocks, the covariance matrix is given by:

$$
\Sigma = \begin{bmatrix}
\operatorname{Var}(X) & \operatorname{Cov}(Y, X) \\
\operatorname{Cov}(X, Y) & \operatorname{Var}(Y)
\end{bmatrix}.
$$

The diagonal contains each stock's own variance. The off-diagonal entries contain the covariance between the stocks. Since $\operatorname{Cov}(X, Y) = \operatorname{Cov}(Y, X)$, the covariance matrix is symmetric.

##### Risk for 2 Stocks

Given two stocks $X$ and $Y$, with weights $w_1$ and $w_2$, their total weighted return is $w_1 X + w_2 Y$. The variance, or risk, of this portfolio is given by:

$$\text{Risk} = \text{Var}(w_1 X + w_2 Y) = w_1^2 \text{Var}(X) + w_2^2 \text{Var}(Y) + 2w_1w_2 \text{Cov}(X, Y)$$

The first two terms measure each individual stock's fluctuation, and the third incorporates how the stocks fluctuate relative to one another.

##### Risk for $n$ Stocks

Let $\vec{w} \in \mathbb{R}^{n}$ be the vector of weights indicating the degree of investment in each stock. Generalizing the formula for two stocks, **portfolio risk** can be expressed as:

$$
\boxed{\operatorname{Risk}=\vec{w}^{\top}\Sigma\vec{w}.}
$$


## Formulating the Constrained Optimization Problem

To formulate an optimization problem as a mathematical program, beyond the data, there are three more essential pieces:

1. **Decision variables:** the optimizer's choices
2. **Objective:** what the optimizer should maximize
3. **Constraints:** what every feasible solution must satisfy

Before reading on, think about how to define these three components for portfolio optimization. **Spend some time brainstorming about the decision variables, in particular.**

<div style="background-color: rgba(30, 144, 255, 0.15); padding: 15px; border-radius: 8px;">

### Decision Variables

For each stock $i$, the optimizer chooses a weight $w_i$: the fraction of the total budget to invest in stock $i$. 

Together, these weights form the portfolio vector $\vec{w} \in {\mathbb R}^n$, where $n$ is the total number of stocks in the data set.

</div>

<div style="background-color: rgba(30, 144, 255, 0.15); padding: 15px; border-radius: 8px;">

### Objective

Your goal is to discover the portfolio with the highest historical return. 

A portfolio's return is the weighted sum of its stocks' historical returns, where the weights are the fraction of the budget invested in each stock.

</div>

<div style="background-color: rgba(30, 144, 255, 0.15); padding: 15px; border-radius: 8px;">

### Constraints

A feasible portfolio must satisfy the following constraints:

- All weights are nonnegative (short selling is not allowed).
- The weights are between 0 and 1, and their sum does not exceed $1$.
- The **portfolio’s risk** does not exceed the chosen maximum risk threshold.

Any portion of the budget that is not invested remains as cash, which has zero historical return and zero modeled risk.

</div>

## Tasks

At this point, you should understand how the data, decision variables, objective, and constraints fit together. In Task 1.1, you will compute the data-derived constants needed by the model; in Task 1.2, you will complete the mathematical formulation of the problem.

### Task 1.1: Calculate Portfolio Statistics

Complete `calculate_stats(data, stock_names)`. 

This function returns a dictionary mapping each selected stock to its historical return, defined as its final normalized price minus 1. It also computes the covariance matrix of the normalized price histories. 

Both outputs must follow the supplied `stock_names` order.

In [ ]:
def calculate_stats(data, stock_names):
    """
    Calculate the historical returns and covariance matrix for the selected stocks.
    
    Input:
        data: Dictionary of normalized price histories keyed by ticker.
        stock_names: List of stock tickers to include in the calculations.
    Output:
        historical_returns: Dictionary mapping each stock ticker to its historical return (final normalized price - 1).
        covariance_matrix: Covariance matrix of the selected stocks' price histories.
    """
    # TODO: Calculate and return the selected stocks' historical returns and covariance matrix.
    # Use np.cov() for the covariance matrix.
    pass

#### View the historical return and covariance matrix of the stocks!

In [ ]:
stock_returns, covariance_matrix = calculate_stats(data, stock_names)
returns = np.array([stock_returns[name] for name in stock_names])

print("Historical Returns for Stocks:")
display(pd.Series(stock_returns, name="Historical return").to_frame())

# Display the covariance matrix of the selected stocks
print("Covariance Matrix of Stocks:")
display(pd.DataFrame(covariance_matrix, index=stock_names, columns=stock_names))

#### Provided Tests for `calculate_stats`

Run this test after completing Task 1.1. It checks returns, covariance, stock selection, and stock ticker ordering.

You can add to this testbed, but we do not recommend editing our tests!!

In [ ]:
def test_calculate_stats():
    """
    Check historical returns, sample covariance, stock selection, and
    ticker ordering for a sample dataset with several price histories.
    """
    sample = {
        "ALPHA": [1, 2, 3, 4, 5],
        "BETA": [1, 4, 3, 2, 1],
        "GAMMA": [1, 1, 3, 3, 2],
        "UNUSED": [1, 10, 10, 10, 10],
    }
    selected = ["GAMMA", "ALPHA", "BETA"]
    historical_returns, covariance = calculate_stats(sample, selected)

    assert list(historical_returns) == selected
    assert historical_returns == {"GAMMA": 1, "ALPHA": 4, "BETA": 0}
    expected_covariance = [
        [1.0, 1.0, 0.0],
        [1.0, 2.5, -0.5],
        [0.0, -0.5, 1.7],
    ]
    np.testing.assert_allclose(covariance, expected_covariance)


test_calculate_stats()
print("calculate_stats test passed.")

### Task 1.2: Mathematical Formulation

Write the mathematical formulation of the portfolio optimization problem. Identify the constants, decision variables, objective function, and constraints. You can write in LaTeX (and submit as a pdf) or on paper by hand (and submit as an image). Please make sure to state the file name in `README.md`.


### Task 1.3: Build and Solve the Model with `CVXPY`

Complete `solve_portfolio(returns, covariance_matrix, maximum_risk)` by translating your mathematical formulation into `CVXPY`. We encourage you to use an LLM for assistance with this task.

We recommend using the CLARABEL solver (i.e., model.solve(solver="CLARABEL")) to solve your `CVXPY` problem formulation.


In [ ]:
def solve_portfolio(returns, covariance_matrix, maximum_risk):
    """
    Build and solve the portfolio optimization model.

    Inputs:
        returns: one historical return per stock
        covariance_matrix: covariance matrix in the same stock order
        maximum_risk: maximum permitted portfolio variance

    Return a dictionary with these keys:
        weights: a one-dimensional array of optimized investment fractions,
            in the same order as returns
        historical_return: optimized portfolio return
        actual_variance: variance of the optimized portfolio
    """
    # TODO: Build and solve the CVXPY model, then return the requested values.
    pass

In [ ]:
# DO NOT EDIT
# This cell sets the maximum risk to 0.01, solves the model, and displays the
# optimized investment fractions, historical return, and actual variance.
result = solve_portfolio(returns, covariance_matrix, 0.01)
weight_values = np.asarray(result["weights"]).reshape(-1)
allocation = dict(zip(stock_names, weight_values))

print(f"Historical return: {result['historical_return']:.1%}")
print(f"Variance: {result['actual_variance']:.6g}")
display(pd.Series(allocation, name="Investment fraction").to_frame())

#### Provided Portfolio Optimization Model Tests

While developing your model, you can run these tests to **Verify** and then **Refine** the LLM-generated or your own code.

You can add to this testbed, but we do not recommend editing our tests!!

In [ ]:
def test_portfolio_model():
    """Check that the model returns the known optimal investment weights."""
    expected_weights = {
        "AAPL": 0.0,
        "JNJ": 0.1064703,
        "JPM": 0.0,
        "KO": 0.0845664,
        "MSFT": 0.0,
        "NVDA": 0.0,
        "PG": 0.0,
        "SPOT": 0.2800798,
        "TSLA": 0.0,
        "V": 0.0,
    }
    test_result = solve_portfolio(returns, covariance_matrix, 0.01)
    actual_allocation = dict(
        zip(stock_names, np.asarray(test_result["weights"]).reshape(-1))
    )
    for stock, expected in expected_weights.items():
        np.testing.assert_allclose(actual_allocation[stock], expected, atol=1e-3)

    print("Expected weights:", expected_weights)
    print("Actual weights:", actual_allocation)


test_portfolio_model()
print("Portfolio model test passed.")

### Task 1.4: What happens to return as you vary risk?

Your final task is to solve the model for three provided risk thresholds: low, medium, and high. For each portfolio your model finds, output its decisions (i.e., its weights), its historical return (i.e., the value of its objective function), and its risk (the portfolio's variance).

In addition, construct an **equal-weighted portfolio** that invests the same fraction of the budget in each stock. Calculate the risk and historical return of this portfolio as well.

In [ ]:
# TODO: Add a fourth baseline that invests the same fraction in every stock (equal-weight portfolio)
equal_weights = ...
equal_historical_return = ...
equal_actual_variance = ...

# DO NOT EDIT below this
# This code solves the model for each supplied risk limit and records each
# portfolio's weights, historical return, and actual variance.
risk_limits = np.array([0.005, 0.01, 0.02])

portfolio_weights = {}
portfolio_returns = {}
summary_rows = []

for label, limit in zip(("Low", "Medium", "High"), risk_limits):
    result = solve_portfolio(returns, covariance_matrix, float(limit))
    values = np.asarray(result["weights"]).reshape(-1).copy()
    historical_return = result["historical_return"]
    actual_variance = result["actual_variance"]
    portfolio_weights[label] = values
    portfolio_returns[label] = historical_return
    summary_rows.append({
        "portfolio": label,
        "maximum_risk": limit,
        "actual_variance": actual_variance,
        "historical_return": historical_return,
    })

# Records the equal-weighted portfolio for the provided plot.
portfolio_weights["Equal-weighted"] = equal_weights
portfolio_returns["Equal-weighted"] = equal_historical_return
summary_rows.append({
    "portfolio": "Equal-weighted",
    "maximum_risk": np.nan,
    "actual_variance": equal_actual_variance,
    "historical_return": equal_historical_return,
})

#### Visualizing the Relationship

After you construct the four portfolios, run the following visualization cell. It will create the comparison table and grouped bar chart for you, listing each portfolio's historical return and actual risk in the legend.

In [ ]:
# DO NOT EDIT

summary = pd.DataFrame(summary_rows).set_index("portfolio")
summary["invested_fraction"] = {
    name: float(np.sum(values)) for name, values in portfolio_weights.items()
}
weight_comparison = pd.DataFrame(portfolio_weights, index=stock_names)
display(summary)
display(weight_comparison)

legend_labels = {
    name: (
        f"{name} (return: {portfolio_returns[name]:.1%}, "
        f"actual risk: {summary.loc[name, 'actual_variance']:.4f})"
    )
    for name in portfolio_weights
}
plot_data = weight_comparison.rename(columns=legend_labels)

ax = plot_data.plot.bar(figsize=(12, 5), rot=45)
ax.set_xlabel("Stock")
ax.set_ylabel("Investment fraction")
ax.legend(title="Portfolio results", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### MCQs

In your `README.md`, answer the following MCQs:

#### Task 1.4.1
**The optimized portfolio invests about 33% of the budget at the low-risk limit and about 67% at the high-risk limit. What best explains this change?**

A) A higher risk limit requires that the optimizer spend the entire budget to counter-balance the fluctuations of high-risk stocks.  
B) Relaxing the risk constraint allows the optimizer to invest more in its preferred stocks, with less of the budget left as cash.  
C) Increasing the risk limit causes every stock's historical return to increase.  
D) The solver always invests a fraction equal to the square root of the risk limit.  

#### Task 1.4.2

**The equal-weighted portfolio has a higher historical return (45.2%) than the optimized medium-risk portfolio (45.0%). Does this make the equal-weighted portfolio a better solution to the medium-risk optimization problem?**

A) Yes, because historical return is the objective.  
B) Yes, because equal weighting automatically satisfies every risk limit.  
C) No, because the equal-weighted portfolio’s risk exceeds the medium-risk limit.  
D) No, because the equal-weighted portfolio does not invest the entire budget.
